## **Flow Matching**

**Intuition**

Flow matching is a generative modeling approach that learns to transform a simple base distribution (e.g. Gaussian noise) $p_0 = \mathcal{N}(0, I)$ into the data distribution $p_1$ by learning a **velocity field** $v_\theta(x, t)$. Generating a sample means starting from noise and **following an Ordinary Differential Equation (ODE)** forward in time:

$$
\frac{dx_t}{dt} = v_\theta(x_t, t),\qquad x_0 \sim \mathcal{N}(0, I),\qquad t \in [0, 1]
$$

The sample at $t=1$ is your generated data point. This is the **continuous normalizing flow (CNF)** picture, but trained without expensive ODE simulation during training.

### **The problem flow matching solves**

Classic CNFs are trained by maximum likelihood, which requires simulating the ODE and computing a divergence (Jacobian trace) at every training step — slow and unstable.

**Flow Matching** instead regresses the velocity field directly against a known target. If we had access to the ideal velocity field $u_t(x)$ that transports $p_0$ to $p_1$, we could just regress onto it:

$$
\mathcal{L}_{\text{FM}} = \mathbb{E}_{t,\ x_t}\big[\ \| v_\theta(x_t, t) - u_t(x_t) \|^2\ \big]
$$

The catch: $u_t$ for the full data distribution is intractable. The key trick is to make it tractable per-sample.

### **Conditional Flow Matching (CFM)**

The insight: instead of the intractable marginal velocity field, we condition on a **single data point** $x_1$ and define a simple per-sample path from noise $x_0$ to $x_1$. The simplest choice is a **straight line** (linear interpolation):

$$
x_t = (1-t)\, x_0 + t\, x_1,\qquad x_0 \sim \mathcal{N}(0, I),\quad x_1 \sim p_{\text{data}}
$$

The target velocity along this straight path is simply the **constant displacement**:

$$
u_t(x_t \mid x_0, x_1) = x_1 - x_0
$$

Remarkably, regressing onto this per-sample conditional target has the **same gradient** as regressing onto the intractable marginal field. So the CFM loss is just:

$$
\mathcal{L}_{\text{CFM}} = \mathbb{E}_{t,\ x_0,\ x_1}\big[\ \| v_\theta(x_t, t) - (x_1 - x_0) \|^2\ \big]
$$

**Intuition:** the network learns, at every point and time, which direction to move to reach the data. It is a simple MSE regression — no ODE solving, no divergence, during training.

### **CFM training step (pseudocode)**

```python
import torch
import torch.nn.functional as F

def cfm_training_step(model, x1):
    # x1: a batch of real data samples
    B = x1.shape[0]

    # sample noise and a random time in [0, 1]
    x0 = torch.randn_like(x1)
    t = torch.rand(B, device=x1.device)
    t_b = t.view(-1, *([1] * (x1.dim() - 1)))   # broadcast over feature dims

    # straight-line interpolation between noise and data
    x_t = (1 - t_b) * x0 + t_b * x1

    # target velocity is the constant displacement
    target = x1 - x0

    # regress the predicted velocity onto the target
    v_pred = model(x_t, t)
    loss = F.mse_loss(v_pred, target)
    return loss
```

**Sampling:** start from `x = torch.randn(...)` and integrate `dx/dt = v_theta(x, t)` from $t=0$ to $t=1$ (e.g. with a simple Euler solver), then return `x`.

### **Rectified Flow**

**Rectified Flow** is the same straight-line (linear interpolation) instance of flow matching, with an extra emphasis: it tries to make the learned trajectories as **straight as possible**. Straight ODE paths can be integrated accurately with very few steps — ideally a single Euler step.

This is achieved via **reflow**: regenerate $(x_0, x_1)$ pairs by running the learned model, then retrain on those re-coupled pairs. Each round straightens the paths further, enabling fast few-step or one-step generation.

### **Flow Matching vs. Diffusion**

| | **Diffusion (DDPM)** | **Flow Matching** |
|---|---|---|
| Framework | Reverse a stochastic noising process (SDE) | Learn a deterministic velocity field (ODE) |
| Network predicts | Noise $\epsilon$ (or score) | Velocity $v$ |
| Probability path | Implied by Gaussian noise schedule (often curved) | Chosen explicitly (straight line in CFM) |
| Sampling | Many stochastic steps (DDIM for fewer) | ODE integration; straight paths allow few steps |
| Training loss | MSE on predicted noise | MSE on predicted velocity |

They are deeply related: diffusion models can be written as a particular probability path, and flow matching generalizes the framework by letting you **choose** the path. The simple, often-straighter paths of CFM tend to train more stably and sample in fewer steps. Modern systems such as Stable Diffusion 3 use flow matching.

### **References**

- Lipman et al., *Flow Matching for Generative Modeling*, 2023
- Liu et al., *Rectified Flow*, 2023
- Albergo & Vanden-Eijnden, *Stochastic Interpolants*, 2023

Videos: [1](https://www.youtube.com/watch?v=DDq_pIfHqLs), [2](https://www.youtube.com/watch?v=rcWMRA9E5RI), [3](https://www.youtube.com/watch?v=i7LjDvsLWCg)